# 텍스트 데이터 다루기 목차
* [Chapter 1 개요](#chapter1)
* [Chapter 2 단어 임베딩 이해하기](#chapter2)
* [Chapter 3 텍스트 토큰화하기](#chapter3)
* [Chapter 4 토큰을 토큰 ID로 변환하기](#chapter4)
* [Chapter 5 특수 문맥 토큰 추가하기](#chapter5)
* [Chapter 6 바이트 페어 인코딩(BPE)](#chapter6)
* [Chapter 7 슬라이딩 윈도우로 데이터 샘플링하기](#chapter7)
* [Chapter 8 토큰 임베딩 만들기](#chapter8)
* [Chapter 9 단어 위치 인코딩하기](#chapter9)

## Chapter 1 개요 <a class="anchor" id="chapter1"></a>
1. 사전 훈련 단계에서 LLM은 텍스트를 한 번에 한 단어씩 처리한다.

2. 다음 단어 예측 작업으로 수백만에서 수십억 개의 파라미터를 가진 LLM을 훈련하려면 훈련 데이터셋을 준비해야한다.
   - 이번 장에서는 첫 번째 스탭인 데이터 샘플링 파이프라인에 초점을 맞춘다.
   - 텍스트를 개별 단어와 부분단어 토큰으로 분활하는 작업
   - LLM을 위해 벡터 표현으로 인코딩 
   - 바이트 페어 인코딩
   - 샘플링과 데이터 로드전략을 구현하여 LLM 훈련에 필요한 이력-출력 쌍 생성

   ![GPT](image/02-00-gpt2.png)
  

## Chapter 2 단어 임베딩 이해하기 <a class="anchor" id="chapter2"></a>
1. LLM을 포함해 심층 신경망 모델은 원시텍스트를 바로 처리할 수 없다.
    - 텍스트는 범주형 데이터(categorical data)로, 모델이 이해할 수 있는 숫자형 데이터로 변환해야한다.

2. 단어를 벡터 형태로 변환하는 개념을 흔히 임베딩(embedding)이라고 한다.
    - 비디오, 오디오, 텍스트와 같은 여러 데이터를 임베딩할 수 있다.
    - 데이터 포멧마다 고유한 임베딩 모델이 필요하다.
    - 임베딩은 단어, 이미지, 심지어 문서 전체와 같이 이산적인 객체를 연속적인 벡터 공간의 한포인트로 맵핑한다.
    - 임베딩의 주요 목적은 비수치 데이터를 신경망이 처리할 수 있는 포멧으로 변환하는 것이다.

        ![embedding](image/02-01-embedding.png)

3. 단어 임배딩이 텍스트 임베딩의 가장 일반적인 형태.
    - 문장, 단락 또는 문서 전체를 위한 임베딩도 있다.

4. 단어 임베딩을 위해 몇 가지 알고리즘과 프레임워크가 개발되었다.
    - Word2Vec
        - 타킷 단어가 주어지면 문맥 단어를 예측하거나 그 반대의 방식으로 신경망을 훈련하여 단어 임베딩 생성
        - 비슷한 맥락에 등장하는 단어는 비슷한 의미를 가지는 경향이 있다
        - 시각화를 위해 2차원 단어 임베딩 공간에 투영하면 비슷한 단어는 서로 가깝게 위치한다.
        - 단어 임베딩 차원이 높을 수록 미묘한 관계를 잘 감지할 수 있지만 효율성이 떨어진다.

            ![word2vec](image/02-02-word2vec2.png)

5. LLM은 일반적으로 Word2Vec을 사용하는 대신 입력층의 일부로 자체적인 임베딩을 만들고 훈련 중에 업데이트 한다.
    - LLM 훈련의 일부로 임베딩을 최적화하면 임베딩을 특정 작업과 주어진 데이터에 최적화 할 수 있다.

6. 사람이 인식하는 감각과 일반적인 그래픽 표현은 3차원 이하로 제한되기 때문에 고차원의 임베딩은 시각화하기 어렵다.
    - 산점도로 임베딩을 표현할 수 있다.
    - GPT-2, GPT-3의 경우 임베딩 크기(은닉 상태)가 각각 768, 12288 고차원의 임베딩을 사용한다.

## Chapter 3 텍스트 토큰화하기 <a class="anchor" id="chapter3"></a>
1. LLM을 위한 임베딩을 만들기 전에 텍스트를 개별 토큰(token)으로 분할하는 작업이 필요하다.
    - 토큰은 단어, 부분단어, 문자 또는 심지어 바이트일 수 있다.
    - 토큰화(tokenization)는 텍스트를 토큰으로 분할하는 과정이다.
    - 토큰화는 자연어 처리(NLP)에서 중요한 전처리 단계이다.

        ![word2vec](image/02-02-token4.png)

2. 이디스 워튼(Edith Wharrton)의 단편 소설 "The Verdict(심판)"을 토큰화 한다.
    - 퍼블릭 도메인에 배포되어있기 때문에 자유롭게 사용할 수 있다.
    - 텍스트가 위키문헌 사이트에 공개(https://wikisource.org/wiki/The_Verdict)되어 있다.

In [1]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [6]:
# 파이썬으로 단편 소설 텍스트 샘플로 읽기
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()
print("총 문자 수:", len(raw_text))
print(raw_text[:99])  # 처음 100자 출력

총 문자 수: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


3. 20,479 문자로 이루어진 이 단편 소설을 개발 단어와 특수 문자로 토큰화 한다.

4. 파이썬 정규 표현식 라이브러리인 re를 사용한다.

5. 토크나이져를 개발할 때 공백을 별도의 문자로 인코딩할지 아니면 삭제할지는 애플리케이션과 요구사항에 따라 다르다.
    - 공백을 제거하면 자원소모가 줄어들고, 유지하면 텍스트의 정확한 구조에 민감한 모델을 훈련할 때 도움이 될 수 있다.
    - 파이썬 코드는 들여쓰기와 공백에 민감하다.

In [7]:
import re
text = "Hello, world. This, is a test."  # 예시 문장
result = re.split(r"[\s]+", text)  # 공백 구분

# 일부 단어는 구두점과 함께 유지
# 예: "Hello,"와 "world."
print(result)  # ['Hello,', 'world.', 'This,', 'is', 'a', 'test.']

# 구두점과 단어를 분리
result = re.split(r'([,.]|\s)', text)  # 공백, 구두점 구분
print(result)  # ['Hello', 'world', 'This', 'is', 'a', 'test', '']

# 공백 제거
result = [item for item in result if item.strip() != '']
print(result)  # ['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']

# 물음표, 따움표, 이 대시와 같은 유형의 특수문자 처리
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)  # 물음표, 따움표, 이 대시와 같은 유형의 특수문자 처리
result = [item for item in result if item.strip() != '']
print(result)  # ['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']

['Hello,', 'world.', 'This,', 'is', 'a', 'test.']
['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']
['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']
['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


6. re를 통해 텍스트를 개별 단어와 구두점 문자로 분할 수 있다.
    - 입력 텍스트: "Hello, world. Is this-- a test?"
    - 10개의 개별 토큰 출력: ['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']

In [8]:
# 이디스 워튼의 소설에 대한 토큰화
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip() != '']
print("총 토큰 수:", len(preprocessed))
print(preprocessed[:30])  # 처음 30개 토큰 출력

총 토큰 수: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Chapter 4 토큰을 토큰 ID로 변환하기 <a class="anchor" id="chapter4"></a>
1. 토큰을 파이썬 문자열에서 정수 표현인 토큰 ID로 변환하기
    - 토큰 ID를 임베딩 벡터로 변화하기 전의 중간 단계

2. 토큰을 토큰 ID로 매핑하려면 어휘사전(vocabulary)이 필요하다.
    - 훈련 세트에 있는 모든 텍스트를 개별 토큰으로 분할한 후 고유한 토큰의 집합을 만든다.
    - 개별 토큰은 알파벳 순서로 정렬되어 있으며 중복 토큰은 삭제된다.
    - 각 토큰에 고유한 정수 인덱스를 할당한다.

        ![tokenid](image/02-03-tokenId.png)

In [9]:
# 토큰화된 이드스 워튼의 소설을 알파벳 순서로 정렬
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("고유 토큰 수:", vocab_size)

고유 토큰 수: 1130


In [10]:
# 어휘사전 만든 후 처음 51개 항목 출력
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


3. 어휘사전은 개별 토큰과 이에 연관된 고유한 정수 레이블을 담고있다.
    - 어휘사전은 훈련세트 전체를 사용해 구축

4. 어휘사전을 새로운 텍스트에 적용하여 토큰 ID로 변환시킨다.

    ![어휘사전](image/02-03-voca.png)

5. LLM의 출력을 숫자에서 텍스트로 변활할 때 토큰 ID를 다시 토큰으로 매핑하는 역어휘사전(reverse vocabulary)이 필요하다.
    - 역어휘사전은 어휘사전의 키-값 쌍을 뒤집어서 만든다.

In [11]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) # 'hello,. world'

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 구둣점 문자 앞의 공백을 삭제합니다.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [12]:
# 소설의 한 구절 토큰화
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print("토큰 ID:", ids)

토큰 ID: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


6. 토크나이저는 일반적으로 인코드 메서드와 디코드 메서드를 구현한다.
    - 인코드 메서드는 텍스트를 토큰 ID의 리스트로 변환한다.
    - 디코드 메서드는 토큰 ID의 리스트를 다시 텍스트로 변환한다.

In [13]:
text = "Hello, do you like tea. Is this-- a test?"

# "Hello"란 단어가 없기 때문에 에러 발생
tokenizer.encode(text)

KeyError: 'Hello'

## Chapter 5 특수 문맥 토큰 추가하기 <a class="anchor" id="chapter5"></a>
1. 특수 토큰은 알지 못하는 단어, 문서 경계 등을 표시하는데 사용된다.
    - 예를 들어, 어휘사전에 없는 단어가 나타나면 일반적으로 <|unk|>라는 특수 토큰으로 대체한다.
    - 관련이 없는 텍스트 사이에 <|endoftext|> 토큰을 삽입하여 모델이 문맥을 구분할 수 있도록 한다.
       - 훈련을 위해 텍스트가 연결되어 있지만 사실 관련이 없다고 알려준다.
    - 특수 토큰은 어휘사전의 일부로 간주되며 고유한 토큰 ID가 할당된다.

In [15]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

# 1130 --> 1132로 어휘사전 확장
print("고유 토큰 수:", len(vocab.items()))

# 마지막 5개의 항목 출력
for item in list(vocab.items())[-5:]:
    print(item)

고유 토큰 수: 1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [16]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) # 'hello,. world'

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # 어휘사전에 없는 단어는 <|unk|>로 대체
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 구둣점 문자 앞의 공백을 삭제합니다.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [17]:
# 어휘 사전에 없는 단어가 포함어 있고, 서로 관련이 없는 2개의 독립된 문장이 연결된 샘플 토큰화
text1 = "Hello, do you like tea?"
text2 = "in the sunlit terraces of the palace."
text = text1 + " <|endoftext|> " + text2
print("샘플 텍스트:", text)

샘플 텍스트: Hello, do you like tea? <|endoftext|> in the sunlit terraces of the palace.


In [18]:
# 토큰화 진행
tokenizer = SimpleTokenizerV2(vocab)
print("토큰 ID:", tokenizer.encode(text))
print("디코딩된 텍스트:", tokenizer.decode(tokenizer.encode(text)))

토큰 ID: [1131, 5, 355, 1126, 628, 975, 10, 1130, 568, 988, 956, 984, 722, 988, 1131, 7]
디코딩된 텍스트: <|unk|>, do you like tea? <|endoftext|> in the sunlit terraces of the <|unk|>.


2. 원본 입력 텍스트와 역토큰화된 텍스트를 비교해 보면 훈련 데이터셋은 "hello"와 "palace"가 없다는 것을 알 수 있다.

3. LLM에 따라 추가적인 토큰을 사용하기도 한다.
    - BOS(beginning of sequence): 텍스트의 시작을 표시
    - EOS(end of sequence): 텍스트의 끝을 표시
    - PAD(padding): 배치 내의 모든 시퀀스가 동일한 길이가 되도록 짧은 시퀀스를 채우는 데 사용
    - CLS(classification): 분류 작업에서 시퀀스의 시작을 나타내는 데 사용

4. GPT 모델이 사용하는 토크나이져는 "<|endoftext|>" 토큰만 사용한다.
    - 알지못하는 토큰이 생성되지 않는다.

## Chapter 6 바이트 페어 인코딩(BPE) <a class="anchor" id="chapter6"></a>
1. 바이트 페어 인코딩(Byte Pair Encoding, BPE)은 어휘사전을 구축하는 데 사용되는 토큰화 알고리즘이다.
    - GPT-2와 GPT-3 모델에서 사용되었다.
    - BPE는 자주 발생하는 문자 쌍을 반복적으로 병합하여 더 큰 단위의 토큰을 만든다.
    - BPE는 어휘사전 크기를 제어하면서도 텍스트의 다양한 표현을 포착할 수 있다.
    - 반복적으로 자주 등장하는 문자를 부분단어로 합치고 다시 자주 등장하는 부분단어를 단어로 합쳐 어휘사전을 구축한다.

2. BPE 구현은 복잡하기 때문에 파이썬 오픈소스 라이브러리인 tiktoken을 사용하는 것이 좋다.
    - tiktoken은 OpenAI에서 개발한 BPE 토크나이져 라이브러리이다.
    - GPT-2, GPT-3, GPT-4 모델에서 사용되는 토크나이져를 지원한다.
    - tiktoken은 빠르고 효율적이며 사용자 정의 어휘사전을 지원한다.
    - tiktoken은 pip를 통해 설치할 수 있다.
    - `pip install tiktoken` 명령어로 설치

In [19]:
# tiktoken 버전 확인
import tiktoken
print(tiktoken.__version__)

0.12.0


In [20]:
# 토크나이져 초기화
tokenizer = tiktoken.get_encoding("gpt2")

In [21]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

# 샘플 텍스트 토큰화
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

# 토큰 ID를 다시 텍스트로 변환
string = tokenizer.decode(integers)
print(string)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


3. "<|endoftext|>"는 어휘사전의 크기는 50257개인데 가장 큰 토큰인 50256이 할당되었다.

4. BPE는 알지 못하는 단어인 "someunknownPlace"를 정확히 인코딩하고 디코딩한다.

5. BPE 알고리즘은 어휘사전에 없는 단어를 더 작은 부분단어로 나누어 처리한다.

    ![어휘사전](image/02-06-bpe3.png)


In [22]:
# 연습문제 2.1
#   - tiktoken 라이브러리의 BPE 토크나이져를 사용하여 다음 문장을 토큰화하고, 다시 텍스트로 역변환하세요.
#     "Akwirw ier"

exercise_text = "Akwirw ier"
# 개별 토큰아이디 출력
print("토큰 ID:", tokenizer.encode(exercise_text))

# 토큰: 토큰 ID 매핑 생성
token_id_mapping = {tokenizer.decode([token_id]): token_id for token_id in tokenizer.encode(exercise_text)}
print("토큰: 토큰 ID 매핑:", token_id_mapping)

# 토큰화된 결과 출력
print("디코딩된 텍스트:", tokenizer.decode(tokenizer.encode(exercise_text)))

토큰 ID: [33901, 86, 343, 86, 220, 959]
토큰: 토큰 ID 매핑: {'Ak': 33901, 'w': 86, 'ir': 343, ' ': 220, 'ier': 959}
디코딩된 텍스트: Akwirw ier


## Chapter 7 슬라이딩 윈도우로 데이터 샘플링하기 <a class="anchor" id="chapter7"></a>
1. 토큰나이져로 입력 값이 토큰으로 변환되었다면 LLM 훈련에 필요한 입력-타깃 쌍을 생성해야한다.
    - LLM 훈련의 목표는 주어진 토큰 시퀀스에서 다음 토큰을 예측하는 것이다.
    - 입력-타깃 쌍은 모델이 이전 토큰을 기반으로 다음 토큰을 예측하는 방법을 학습하는 데 사용된다.

2. 슬라이딩 윈도우
    - 텍스트 샘플에서 입력 불록을 추출한다.
    - LLM은 입력 불록의 다음에 오는 단어를 예측하도록 훈련된다.
    - 훈련하는 동안 타깃 이후의 단어는 모두 마스킹된다.

        ![슬라이딩 윈도우](image/02-06-window.png)

In [23]:
# 입력-타깃 쌍을 추출하는 데이터 로더 구현.
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

enc_text = tokenizer.encode(raw_text, allowed_special={"<|endoftext|>"})
print("총 토큰 수:", len(enc_text))


총 토큰 수: 5145


In [24]:
# 조금 더 흥미로운 구성을 만들기 위해 처음 50개 토큰 삭제
enc_sample = enc_text[50:]

# 입력-타깃 쌍 생성
# 입력 토큰을 담을 x, 입력에서 토큰 하나만큼 이동한 타깃을 담을 y 생성
context_size = 4 # 입력에 얼마나 많은 토큰을 포함할지 결정
x = enc_sample[:context_size] # 처음 4개 토큰
y = enc_sample[1:context_size+1] # 2번째 토큰부터 5번째 토큰
print("입력 토큰:", x)
print("타깃 토큰:", y)
print("입력 텍스트:", tokenizer.decode(x))
print("타깃 텍스트:", tokenizer.decode(y))

입력 토큰: [290, 4920, 2241, 287]
타깃 토큰: [4920, 2241, 287, 257]
입력 텍스트:  and established himself in
타깃 텍스트:  established himself in a


In [25]:
# 입력과 토큰 하나만큼 이동시킨 타깃을 사용해 다음 단어 예측을 구성
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    
    # "-->" 왼쪽의 모든 값이 LLM이 받을 입력 값
    # "-->" 오른쪽의 값이 LLM이 예측해야 하는 타깃 값
    print(f"입력 토큰 {i}개: {context} --> 타깃 토큰: {desired} ({tokenizer.decode([desired])})")

입력 토큰 1개: [290] --> 타깃 토큰: 4920 ( established)
입력 토큰 2개: [290, 4920] --> 타깃 토큰: 2241 ( himself)
입력 토큰 3개: [290, 4920, 2241] --> 타깃 토큰: 287 ( in)
입력 토큰 4개: [290, 4920, 2241, 287] --> 타깃 토큰: 257 ( a)


3. 입력 데이터셋을 순회하면서 파이토치 텐서로 입력과 타깃을 반환하는 데이터로더 구현
   - 파이토치 텐서는 일종의 다차원 배열이다.
   - 입력을 담은 텐서와 타깃을 담을 텐서로 구성된다.
   - 아래의 이미지는 토큰 ID를 보여주어야 하지만 편의를 위해 텍스트로 표현한다.
   - x에 입력을 모은다. 각 행은 하나의 입력 문맥을 나타낸다.
   - y에 x에 상응하는 예측 타깃을 모은다. x에서 한 토큰만큼 이동하여 생성된다.

      ![파이토치 텐서](image/02-06-tensor.png)

In [ ]:
# 배치 입력과 타깃을 위한 데이터셋 생성
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"}) # 전체 텍스트를 토큰화

        # 슬라이딩 윈도우를 사용해 책을 max_length 길이의 중접된 시퀸스로 나눈다.
        for i in range(0, len(token_ids) - max_length, stride): 
            input_chunk = token_ids[i:i+max_length] # 입력 시퀀스
            target_chunk = token_ids[i+1:i+max_length+1] # 타깃 시퀀스

            # 텐서로 변환하여 리스트에 추가
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # 전체 데이터셋 크기 반환
    def __len__(self):
        return len(self.input_ids)
    
    # 특정 인덱스의 입력과 타깃 시퀀스 반환
    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "target_ids": self.target_ids[idx]}

In [27]:
# 입력-아킷 쌍의 배치를 생성하기 위한 데이터 로더 생성
def create_dataloader_v1(text, batch_size=4, max_length=256, stride=128, suffle=True, drop_last=True, num_workers=0):
   # 토큰나이져 초기화
   tokenizer = tiktoken.get_encoding("gpt2")

   # 데이터셋 생성
   dataset = GPTDatasetV1(text, tokenizer, max_length, stride)

   # 데이터로더 생성
   dataloader = DataLoader(
       dataset, 
       batch_size=batch_size, 
       shuffle=suffle, 
       drop_last=drop_last, # True로 설정하면 batch_size보다 작은 마지막 배치를 버립니다. 
       num_workers=num_workers # 전처리에 사용할 CPU 프로세스 개수
    )

   return dataloader

In [ ]:
# 문맥 크기 4, 배치 크기 1로 dataloader 테스트
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()
    
dataloader = create_dataloader_v1(
    raw_text, 
    batch_size=1, 
    max_length=4, 
    stride=1, 
    suffle=False, 
    drop_last=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)

# first_batch 변수는 2개의 덴서를 담고 있다.
#    - 하나는 입력 토큰 ID, 다른 하나는 타깃 토큰 ID
#    - max_length가 4이므로 각 텐서의 크기는 [1, 4]이다.
print("입력 토큰 ID:", first_batch["input_ids"])
print("타깃 토큰 ID:", first_batch["target_ids"])

입력 토큰 ID: tensor([[  40,  367, 2885, 1464]])
타깃 토큰 ID: tensor([[ 367, 2885, 1464, 1807]])


In [29]:
# stride=1 테스트
second_batch = next(data_iter)
print("두 번째 배치의 입력 토큰 ID:", second_batch["input_ids"])
print("두 번째 배치의 타깃 토큰 ID:", second_batch["target_ids"])

두 번째 배치의 입력 토큰 ID: tensor([[ 367, 2885, 1464, 1807]])
두 번째 배치의 타깃 토큰 ID: tensor([[2885, 1464, 1807, 3619]])


4. 첫 번째와 두 번째 배치를 비교
    - 두 번째 배치의 토큰 ID가 첫 번째 배치의 토큰 ID에서 한 토큰씩 밀려난 것을 볼 수 있다.
    - stride 변수는 슬라이딩 윈도우가 배치에 걸쳐 입력 위를 이동하는 크기를 지정한다.
    - 스트라이드가 입력 윈도우와 크기가 같다면 배치 사이에 중첩되는 토큰이 없다.

In [36]:
# 연습문제 2.2
#   - 여러 가지 스트라이드와 문맥 크기를 가진 데이터 로더 만들기
#   - max_length=2, stride=2
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()
dataloader = create_dataloader_v1(
    raw_text, 
    batch_size=1, 
    max_length=2, 
    stride=2, 
    suffle=False, 
    drop_last=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("max_length=2, stride=2 입력 토큰 ID:", first_batch["input_ids"])
print("max_length=2, stride=2 타깃 토큰 ID:", first_batch["target_ids"])

#   - max_length=8, stride=2
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()
dataloader = create_dataloader_v1(
    raw_text, 
    batch_size=4, 
    max_length=8, 
    stride=2, 
    suffle=False, 
    drop_last=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("max_length=8, stride=2 입력 토큰 ID:", first_batch["input_ids"])
print("max_length=8, stride=2 타깃 토큰 ID:", first_batch["target_ids"])



max_length=2, stride=2 입력 토큰 ID: tensor([[ 40, 367]])
max_length=2, stride=2 타깃 토큰 ID: tensor([[ 367, 2885]])
max_length=8, stride=2 입력 토큰 ID: tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]])
max_length=8, stride=2 타깃 토큰 ID: tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899],
        [ 1464,  1807,  3619,   402,   271, 10899,  2138,   257],
        [ 3619,   402,   271, 10899,  2138,   257,  7026, 15632],
        [  271, 10899,  2138,   257,  7026, 15632,   438,  2016]])


5. 작은 배치 크기는 훈련 과장에서 메모리를 덜 필요로 하지만 모델 업데이트에 잡음이 더 많이 든다.
    - 큰 배치 크기는 메모리를 더 많이 필요로 하지만 모델 업데이트가 더 안정적이다.
    - 일반적으로 배치 크기는 GPU 메모리 용량에 따라 결정된다.
    - 배치 크기가 클수록 훈련 속도가 빨라질 수 있지만, 메모리 부족으로 인해 훈련이 중단될 수 있다.
    - 배치 크기를 선택할 때는 모델의 복잡성, 데이터셋 크기, 하드웨어 제약 등을 고려해야 한다.

In [ ]:
# 스트라이드와 슬라이딩 윈도우의 크기를 같게하여 배치 사이에 중첩되는 토큰이 없도록 설정
#   - 과대적합을 피할 수 있다.
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()
dataloader = create_dataloader_v1(
    raw_text, 
    batch_size=8, 
    max_length=4, 
    stride=4, 
    suffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("max_length=2, stride=2 입력 토큰 ID:", first_batch["input_ids"])
print("max_length=2, stride=2 타깃 토큰 ID:", first_batch["target_ids"])

max_length=2, stride=2 입력 토큰 ID: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
max_length=2, stride=2 타깃 토큰 ID: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Chapter 8 토큰 임베딩 만들기 <a class="anchor" id="chapter8"></a>
1. LLM 훈련을 위한 입력 텍스트 준비의 마지막 단계는 토큰 ID를 임베딩 벡터로 변환하는 것이다.
   - 준비 단계에서는 이런 임베딩 벡터를 랜덤한 값으로 초기화한다.
   - LLM을 훈련하면서 임베딩 벡터를 최적화한다.

   ![임베딩](image/02-08-embedding.png)

2. GPT와 같은 LLM은 역전파(backpropagation) 알고리즘으로 훈련되는 심층 신경망이므로 연속적인 벡터 푠현인 임베딩이 필수적이다.

In [39]:
# 토큰 아이디: 2, 3, 5, 1
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6  # 어휘사전 크기
output_dim = 3 # 임베딩 벡터의 차원

# 파이토치 임베딩 층 초기화
torch.manual_seed(123)  # 재현성을 위해 시드 설정
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# 가중치 행: 6 / 가중치 열: 3
# 어휘사전에 있는 6개의 토큰 각각에 하나의 행이 할당된다.
# 3개의 임베딩 차원 각각에 하나의 열이 할당된다.
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [ ]:
# 네 번째 행의 값과 같다.
#   - 파이썬 행령을 0부터 시작하므로 인덱스 3에 해당하는 행이 네 번째 행이다.
# 임베딩 층은 토큰 ID를 기반으로 가중치 행렬에서 행을 추출하는 검색 연산을 수행한다.
print(embedding_layer(torch.tensor([3])))

# 어휘사전에 없는 토큰 ID를 사용하면 에러 발생
print(embedding_layer(torch.tensor([7])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


IndexError: index out of range in self

6. 임베딩 층은 본질적으로 원-핫 인코딩 다음에 완전 연결 층을 두어 행렬 곱셈을 수행하는 것을 효율적으로 구현한 것일 뿐이다.
    - 가중치 행렬에서 토큰 ID에 해당하는 임베딩 벡터를 추출하는 룩업 연산을 수행한다.

    ![토치 텐서](image/02-08-torchTensor2.png)

In [ ]:
# 4개의 토큰 ID를 가진 입력 시퀀스에 대한 임베딩 벡터 생성
input_ids = torch.tensor([2, 3, 5, 1])
embeddings = embedding_layer(input_ids)

# 4 * 3 크기의 행렬
print(embeddings)
print("임베딩 벡터의 크기:", embeddings.shape)  

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)
임베딩 벡터의 크기: torch.Size([4, 3])


7. 토큰 -> 토큰 ID -> 토크나이져가 수행

8. 토큰ID -> 단어 임베딩 -> LLM 모델이 수행

## Chapter 9 단어 위치 인코딩하기 <a class="anchor" id="chapter9"></a>
1. 토큰 임베딩의 단점은 시퀸스 안의 토큰 위치 또는 순서에 대한 개념이 셀프 어텐션 메커니즘에 없다.

    ![토치 텐서](image/02-08-torchTensor2.png)

2. 독립적인 토큰에 위치 정보를 주입하는 것은 LLM 학습에 도움이 된다.
    - 예를 들어, "The cat sat on the mat."과 "On the mat sat the cat."은 동일한 단어를 포함하지만 의미가 다르다.
    - 토큰 임베딩에 위치 정보를 추가하면 모델이 단어 순서의 중요성을 이해하는 데 도움이 된다.
    - LLM이 토큰 사이의 순서와 관계를 이해하는 능력을 보강하여 맥락을 고려한 예측을 만들게 한다.

3. 절대위치 임베딩    
    - 입력 시퀸스의 각 위치에 대해서 고유한 임베딩이 토큰에 더해져 정확한 위치 정보를 추가한다.
    - 위치 임베딩 벡터는 원본 토큰 임베딩과 동일한 차원을 갖는다.

        ![위치 임베딩](image/02-09-posembedding.png)

4. 상대위치 임베딩
    - 절대 위치 대신 토큰 간의 상대적 거리를 인코딩한다.
    - 예를 들어, "The cat sat on the mat."에서 "cat"과 "sat" 사이의 거리는 1이다.
    - 멀리 떨어져 있는 정도를 바탕으로 관계를 학습한다.
    - 상대 위치 임베딩은 문맥에 따라 단어의 의미가 달라질 수 있는 언어의 특성을 더 잘 포착할 수 있다.

5. GPT 모델은 훈련 과정에서 최적화되는 절대 위치 임베딩을 사용한다.
    - 최적화 과정은 모델 훈련의 일부로 수행된다.

In [53]:
# 입력 토큰을 256 차원 벡터 표현으로 인코딩한다.
vocab_size = 50257  # GPT-2 어휘사전 크기
output_dim = 256    # 임베딩 벡터의 차원
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# 256 차원의 임베딩 벡터를 가진 50257개의 토큰
print(token_embedding_layer)

with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

# 배치 크기: 8, 문맥 크기: 4, 스트라이드: 4    
dataloader = create_dataloader_v1(
    raw_text, 
    batch_size=8, 
    max_length=4, 
    stride=4, 
    suffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("max_length=2, stride=2 입력 토큰 ID:", first_batch["input_ids"])
print("max_length=2, stride=2 타깃 토큰 ID:", first_batch["target_ids"])
print("입력 토큰 ID 크기:", first_batch["input_ids"].shape)  # [8, 4]q

Embedding(50257, 256)
max_length=2, stride=2 입력 토큰 ID: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
max_length=2, stride=2 타깃 토큰 ID: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
입력 토큰 ID 크기: torch.Size([8, 4])


In [54]:
# 256 차원의 임베딩 벡터로 변환
token_embeddings = token_embedding_layer(first_batch["input_ids"])
print("임베딩 벡터 크기:", token_embeddings.shape)  # [8, 4, 256]

임베딩 벡터 크기: torch.Size([8, 4, 256])


In [ ]:
# 절대 임베딩 추가
context_length = 4

# 256 차원의 임베딩 벡터
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# 위치 임베딩 벡터 생성
#   - torch.arange(context_length)는 [0, 1, 2, ....] 최대 입력 길이 -1까지의 시퀸스를 담고있다 
pos_embedding = pos_embedding_layer(torch.arange(context_length))
print("위치 임베딩 벡터 크기:", pos_embedding.shape)  # [4, 256]

위치 임베딩 벡터 크기: torch.Size([4, 256])


In [56]:
# 토큰임베딩에 위치 임베딩을 더한다.
input_embeddings = token_embeddings + pos_embedding
print("입력 임베딩 벡터 크기:", input_embeddings.shape)  #

입력 임베딩 벡터 크기: torch.Size([8, 4, 256])
